# GNN Model to predict charge for $N_2$ on $Fe(111)$

In [1]:
from mlcolvar.cvs import RegressionCV
from mlcolvar.data import DictModule
from mlcolvar.data.utils import save_dataset, load_dataset
from mlcolvar.utils.trainer import MetricsCallback
from mlcolvar.utils.plot import plot_metrics
from mlcolvar.utils.io import create_dataset_from_trajectories
from mlcolvar.core.nn.graph.schnet import SchNetModel

from lightning import Trainer
from torch import no_grad
from torch.jit import load
from torch.optim import lr_scheduler


from ase.io import read, write

import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm

In [2]:
import os
os.chdir("OPES_2")

## Create Dataset

In [ ]:
sampling_traj_with_traget = np.arange(0, 200001, 10)
rng = np.random.default_rng(42)
sampling = rng.choice(len(sampling_traj_with_traget), size=1000, replace=False)
# sampling = np.arange(100)
np.random.shuffle(sampling)
sampling_traj = sampling_traj_with_traget[sampling]

trajectories = []
for i, index in tqdm(enumerate(sampling)):
    structure = read("traj_comp.traj", index)
    structure.set_pbc([True, True, True])
    trajectories.append(structure)

q = np.loadtxt("CHARGES")[sampling]
target = (q[:,72]+q[:,73])/2

# write("traj.xyz", trajectories)
# np.save("q", q)
# np.save("target", target)

In [3]:
trajectories = read("traj.xyz", ":")
q = np.load("q.npy")
target = np.load("target.npy")

In [ ]:
dataset = create_dataset_from_trajectories(
    "traj.xyz",
    topologies=None,
    cutoff=4.5,
    graph_labels=target.tolist(),
    node_labels=q.tolist(),
    system_selection="type N",
    environment_selection="not type N",
)

# save_dataset(dataset, "dataset")

In [4]:
dataset = load_dataset("dataset")

In [5]:
datamodule = DictModule(dataset, batch_size=16)

## Create Model

In [6]:
gnn_model = SchNetModel(
    n_out=1,
    cutoff=dataset.metadata["cutoff"],
    atomic_numbers=dataset.metadata["atomic_numbers"],
)

options = {
    'optimizer': {'lr': 1e-3},
    'lr_scheduler': {
        'scheduler': lr_scheduler.ExponentialLR,
        'gamma': 0.9999
    }
}

model = RegressionCV(gnn_model, options=options)

/home/gguiard@iit.local/miniforge3/envs/md_env/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['model'])`.


## Train Model

In [7]:
metrics = MetricsCallback()

trainer = Trainer(
    callbacks=[metrics],
    logger=False,
    enable_checkpointing=False,
    max_epochs=10000,
    enable_model_summary=False,
    enable_progress_bar=False,
)

trainer.fit(model, datamodule)

ax = plot_metrics(
    metrics.metrics,          
    keys=['train_loss_epoch','valid_loss'],
    linestyles=['-.','-'],
    colors=['fessa1','fessa5'],
    yscale='log',
)

# model.to_torchscript('../model.ptc', method="trace")
# plt.savefig('plot_metrics.svg')

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA RTX A4000') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/gguiard@iit.local/miniforge3/envs/md_env/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/gguiard@iit.local/miniforge3/envs/md_env/lib/pyth

SystemExit: 1

/home/gguiard@iit.local/miniforge3/envs/md_env/lib/python3.11/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
model = load("model.ptc")

## Make Plots

In [ ]:
model.eval()

In [ ]:
with no_grad():
    pred = model(dataset.get_graph_inputs()).detach().squeeze()

min_value = min(min(target), min(pred))
max_value = max(max(target), max(pred))

fig, ax = plt.subplots(layout="constrained")
ax.plot([min_value, max_value], [min_value, max_value], c='k', ls='--')
ax.scatter(target, pred, s=2)
ax.set_xlabel("Reference")
ax.set_ylabel("Prediction")
ax.set_aspect('equal', 'box')

# fig.savefig("pred.svg")

In [ ]:
train_indices = datamodule._dataset_split[0].indices
valid_indices = datamodule._dataset_split[1].indices

ref_train = target[train_indices]
ref_valid = target[valid_indices]

with no_grad():
    pred = model(dataset.get_graph_inputs()).detach().squeeze()

pred_train = pred[train_indices]
pred_valid = pred[valid_indices]

min_value = min(min(ref_valid), min(ref_train), min(pred_valid), min(pred_train))
max_value = max(max(ref_valid), max(ref_train), max(pred_valid), max(pred_train))

fig, ax = plt.subplots(layout='constrained')
ax.plot([min_value, max_value], [min_value, max_value], color='k', linestyle='--')
ax.scatter(ref_train, pred_train, s=2, label="train")
ax.scatter(ref_valid, pred_valid, s=2, label="valid")

ax.set_xlabel("Reference")
ax.set_ylabel("Prediction")
ax.legend()
ax.set_aspect('equal', 'box')

# fig.savefig("pred_trainvalid.svg")